# SQL reference 7: types, casting, numbers and strings

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-reference-07-types-casting-and-strings.ipynb)

Spark SQL types, `CAST`/`::`/`try_cast` and formatted-number parsing, division and rounding rules (with the differences from other engines), string functions, regular expressions, formatting and hashing, and Spark 4.0 collations. Practise it in [SQL practice 7: types, casting, numbers and strings](sql-practice-07-types-casting-and-strings.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and three helpers: `show(query)` runs a query and prints its rows untruncated, `plan(query)` returns what `EXPLAIN` prints, and `expected_error()` prints the error class of an example that fails on purpose.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Contents

| section | entries |
| --- | --- |
| [Types and casting](#types-and-casting) | [Spark SQL data types](#spark-sql-data-types), [`CAST`, `::` and `TRY_CAST`](#cast--and-try_cast) |
| [Numbers](#numbers) | [Arithmetic and division](#arithmetic-and-division), [Rounding](#rounding) |
| [Strings](#strings) | [Core string functions](#core-string-functions), [Regular expressions](#regular-expressions), [Formatting, hashing and encoding](#formatting-hashing-and-encoding), [Collations and case-insensitive comparison](#collations-and-case-insensitive-comparison) |

## Types and casting

### Spark SQL data types

| type | holds | notes |
| --- | --- | --- |
| `TINYINT`, `SMALLINT`, `INT`, `BIGINT` | 1, 2, 4, 8-byte integers | overflow raises an error in ANSI mode |
| `DECIMAL(p, s)` | exact numbers, `p` digits with `s` after the point (max 38) | use for money |
| `FLOAT`, `DOUBLE` | binary floating point | fast, inexact: `0.1 + 0.2 <> 0.3` |
| `STRING` (`VARCHAR(n)`, `CHAR(n)`) | text | `VARCHAR`/`CHAR` lengths are enforced on table writes only |
| `BOOLEAN` | TRUE, FALSE, NULL | |
| `DATE`, `TIMESTAMP`, `TIMESTAMP_NTZ` | calendar date; instant in the session time zone; wall-clock time without zone | see [dates](sql-reference-08-dates-and-time-series.ipynb) |
| `BINARY` | bytes | |
| `ARRAY<T>`, `MAP<K, V>`, `STRUCT<...>`, `VARIANT` | nested and semi-structured values | see [arrays, structs and JSON](sql-reference-09-arrays-structs-and-json.ipynb) |

`typeof(expr)` returns the type name of any expression; `DESCRIBE t` lists a table's columns.

In [ ]:
show("SELECT typeof(1) AS int_lit, typeof(1.5) AS decimal_lit, typeof(1.5D) AS double_lit, typeof(1L) AS bigint_lit, typeof(amount) AS amount FROM orders LIMIT 1")
show("DESCRIBE orders")

### `CAST`, `::` and `TRY_CAST`

| form | on failure |
| --- | --- |
| `CAST(x AS type)` / `x::type` (Spark 4.0, Postgres, Snowflake, DuckDB) | error in ANSI mode (the Spark 4.0 default) |
| `try_cast(x AS type)` / `x ?::type` | NULL |
| implicit coercion | Spark widens types to a common type in comparisons, `UNION`, `CASE` and function calls; ANSI mode forbids lossy implicit string-to-number conversions |

**Notes:** casting a string to a number trims spaces but rejects thousands separators: `to_number('1,200.00',
'9,999.99')` (or `try_to_number`) parses formatted numbers. Casting a `DOUBLE` to `DECIMAL` rounds; casting to an
integer type truncates toward zero.

In [ ]:
show("SELECT CAST(' 42 ' AS INT) AS trimmed, '7'::INT * 2 AS shorthand, try_cast('4x' AS INT) AS bad, to_number('1,200.00', '9,999.99') AS formatted, CAST(-2.7 AS INT) AS truncated")
with expected_error():
    show("SELECT CAST('4x' AS INT)")

## Numbers

### Arithmetic and division

| operator | Spark result | elsewhere |
| --- | --- | --- |
| `int / int` | `DOUBLE` (`7 / 2 = 3.5`) | Postgres, SQL Server, Snowflake (integers) truncate: `7 / 2 = 3` |
| `a div b` | integer division (`7 div 2 = 3`) | |
| `a % b` / `mod(a, b)` | remainder with the sign of `a` | |
| `pmod(a, b)` | non-negative remainder | useful for bucketing negative keys |
| `decimal * decimal` | precision and scale add up (`DECIMAL(12,2) * DECIMAL(12,2)` is `DECIMAL(25,4)`) | |

**Notes:** porting SQL between engines often breaks on integer division: write `CAST(a AS DECIMAL) / b` or `a * 1.0 /
b` when you mean exact division. Integer overflow raises `ARITHMETIC_OVERFLOW` in ANSI mode (`try_add` returns NULL).

In [ ]:
show("SELECT 7 / 2 AS slash, 7 div 2 AS div, -7 % 3 AS mod, pmod(-7, 3) AS pmod, typeof(amount * amount) AS product_type FROM orders LIMIT 1")
show("SELECT 0.1D + 0.2D = 0.3D AS doubles_equal, 0.1 + 0.2 = 0.3 AS decimals_equal")

### Rounding

| function | does |
| --- | --- |
| `round(x, d)` | round half **up** (away from zero) to `d` digits; negative `d` rounds left of the point (`round(1234.5, -2) = 1200`) |
| `bround(x, d)` | round half **even** (banker's rounding), unbiased over many values |
| `ceil(x)`, `floor(x)` | toward +∞ / −∞; Spark 3.3+ accepts a scale: `ceil(x, 1)` |
| `trunc(x, d)` / cast to integer | toward zero |
| `format_number(x, d)` | a string with thousands separators |

In [ ]:
show("SELECT round(2.5) AS r, bround(2.5) AS br, round(-2.5) AS r_neg, round(1234.567, -2) AS hundreds, ceil(12.01) AS c, floor(-1.5) AS f, format_number(1180.5, 2) AS formatted")

## Strings

### Core string functions

| function | returns |
| --- | --- |
| `length(s)` / `char_length`, `octet_length(s)` | characters / bytes |
| `lower`, `upper`, `initcap` | case changes (`initcap` capitalises each word) |
| `trim(s)`, `ltrim`, `rtrim`, `btrim(s, chars)` | strip spaces or given characters |
| `lpad(s, n, pad)`, `rpad` | pad to length `n` (truncates longer strings) |
| `substring(s, pos [, len])` / `substr`, `left(s, n)`, `right(s, n)` | slices; positions start at **1** |
| `instr(s, sub)`, `locate(sub, s [, pos])`, `position(sub IN s)` | 1-based position, 0 when absent |
| `replace(s, from, to)`, `translate(s, from_chars, to_chars)` | substring replacement / per-character mapping |
| `split(s, regex [, limit])`, `split_part(s, delim, n)`, `substring_index(s, delim, n)` | array / n-th field / prefix before the n-th delimiter |
| `concat(...)`, `concat_ws(sep, ...)`, `\|\|` | concatenation (`concat_ws` skips NULLs) |
| `repeat`, `reverse`, `ascii`, `chr` | misc |

**Notes:** `split` takes a **regular expression** (`split(s, '.')` splits on every character; use `'\\.'`);
`split_part` takes a literal delimiter.

In [ ]:
show("""
SELECT name, length(name) AS len, upper(name) AS up, lpad(CAST(customer_id AS STRING), 5, '0') AS padded,
       substring(name, 1, 4) AS first4, instr(name, ' ') AS space_at, split_part(name, ' ', 2) AS second_word,
       replace(name, ' ', '_') AS snake
FROM customers WHERE customer_id <= 3 ORDER BY customer_id
""")
show("SELECT split('a.b.c', '\\\\.') AS split_regex, split_part('a.b.c', '.', 2) AS part, substring_index('a.b.c', '.', 2) AS prefix, translate('2026-01-05', '-', '/') AS translated")

### Regular expressions

| function | returns |
| --- | --- |
| `s RLIKE pattern` / `regexp_like(s, pattern)` | whether `pattern` matches anywhere |
| `regexp_extract(s, pattern, group)` | the given capture group of the first match (`''` when there is no match) |
| `regexp_extract_all(s, pattern, group)` | an array of every match's group |
| `regexp_substr(s, pattern)` | the first match, NULL when none |
| `regexp_replace(s, pattern, replacement)` | replace every match; `$1` refers to groups |
| `regexp_count(s, pattern)` | number of matches |

**Notes:** Spark uses Java regex. Inside a SQL string literal a backslash must be doubled (`'\\d+'`), and inside a
Python string holding SQL it is doubled again, or use raw strings. Spark returns `''`, not NULL, from
`regexp_extract` when nothing matches: test with `RLIKE` or use `regexp_substr`.

In [ ]:
show(r"""
SELECT note,
       regexp_extract(note, '#(\\d+)', 1) AS first_order,
       regexp_extract_all(note, '#(\\d+)', 1) AS all_orders,
       regexp_replace(note, '\\d', '*') AS masked,
       regexp_count(note, '#\\d+') AS mentions
FROM VALUES ('refund #12 and #9'), ('no order mentioned') AS t(note)
""")

### Formatting, hashing and encoding

| function | returns |
| --- | --- |
| `format_string(fmt, ...)` / `printf` | Java `String.format` formatting (`%05d`, `%.2f` with a `DOUBLE`, `%s`) |
| `md5(s)`, `sha1(s)`, `sha2(s, 256)` | hex digests: stable surrogate keys, change detection, pseudonymisation (add a secret salt for personal data) |
| `hash(...)`, `xxhash64(...)` | fast non-cryptographic hashes (bucketing, sampling) |
| `base64(bin)`, `unbase64(s)`, `encode(s, charset)`, `decode(bin, charset)` | encodings |
| `parse_url(url, part [, key])`, `url_decode(s)` | URL parts and query parameters |
| `uuid()` | a random UUID string (non-deterministic) |

In [ ]:
show("""
SELECT format_string('%05d', customer_id) AS code, sha2(lower(name), 256) AS name_sha256,
       pmod(xxhash64(customer_id), 4) AS bucket, parse_url('https://shop.example/p?utm_source=mail&id=7', 'QUERY', 'utm_source') AS utm
FROM customers WHERE customer_id <= 2 ORDER BY customer_id
""")

### Collations and case-insensitive comparison

Spark 4.0 adds **collations**: `s COLLATE UTF8_LCASE` compares, groups and sorts case-insensitively (`UNICODE_CI`
also ignores accents and case per the Unicode rules). The traditional, portable approach is to normalise both sides
with `lower(trim(s))`.

In [ ]:
show("""
SELECT count(DISTINCT name) AS exact, count(DISTINCT name COLLATE UTF8_LCASE) AS case_insensitive,
       count(DISTINCT lower(trim(name))) AS normalised
FROM VALUES ('Acme Corp'), ('ACME CORP'), ('acme corp '), ('Beta') AS t(name)
""")

**Next:** practise these entries in [SQL practice 7: types, casting, numbers and strings](sql-practice-07-types-casting-and-strings.ipynb).